TRAINING OUR MODEL

In [31]:
import os
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('combined_data.csv')
df.head()


,Angle,CurrentLapTime,Damage,DistanceFromStart,DistanceCovered,FuelLevel,Gear,LastLapTime,Opponent_1,Opponent_2,...,WheelSpinVelocity_1,WheelSpinVelocity_2,WheelSpinVelocity_3,WheelSpinVelocity_4,Z,Acceleration,Braking,Clutch,Gear,Steering
0,0.0,-0.982,0.0,6201.46,0.0,94.0000,0,0.0,16.9999,200.0,...,0.0,0.0,0.0,0.0,0.345263,1.0,0.0,0.64,1,-0.028559
1,0.0,-0.962,0.0,6201.46,0.0,94.0000,0,0.0,16.9999,200.0,...,0.0,0.0,0.0,0.0,0.345263,1.0,0.0,0.64,1,-0.028559
2,0.0,-0.942,0.0,6201.46,0.0,93.9999,0,0.0,16.9999,200.0,...,0.0,0.0,0.0,0.0,0.345263,1.0,0.0,0.64,1,-0.028559
3,0.0,-0.922,0.0,6201.46,0.0,93.9998,0,0.0,16.9999,200.0,...,0.0,0.0,0.0,0.0,0.345263,1.0,0.0,0.64,1,-0.028559
4,0.0,-0.902,0.0,6201.46,0.0,93.9997,0,0.0,16.9999,200.0,...,0.0,0.0,0.0,0.0,0.345263,1.0,0.0,0.64,1,-0.028559


In [33]:
df.columns = [col.strip() for col in df.columns]
df.columns.tolist()

['Angle',
 'CurrentLapTime',
 'Damage',
 'DistanceFromStart',
 'DistanceCovered',
 'FuelLevel',
 'Gear',
 'LastLapTime',
 'Opponent_1',
 'Opponent_2',
 'Opponent_3',
 'Opponent_4',
 'Opponent_5',
 'Opponent_6',
 'Opponent_7',
 'Opponent_8',
 'Opponent_9',
 'Opponent_10',
 'Opponent_11',
 'Opponent_12',
 'Opponent_13',
 'Opponent_14',
 'Opponent_15',
 'Opponent_16',
 'Opponent_17',
 'Opponent_18',
 'Opponent_19',
 'Opponent_20',
 'Opponent_21',
 'Opponent_22',
 'Opponent_23',
 'Opponent_24',
 'Opponent_25',
 'Opponent_26',
 'Opponent_27',
 'Opponent_28',
 'Opponent_29',
 'Opponent_30',
 'Opponent_31',
 'Opponent_32',
 'Opponent_33',
 'Opponent_34',
 'Opponent_35',
 'Opponent_36',
 'RacePosition',
 'RPM',
 'SpeedX',
 'SpeedY',
 'SpeedZ',
 'Track_1',
 'Track_2',
 'Track_3',
 'Track_4',
 'Track_5',
 'Track_6',
 'Track_7',
 'Track_8',
 'Track_9',
 'Track_10',
 'Track_11',
 'Track_12',
 'Track_13',
 'Track_14',
 'Track_15',
 'Track_16',
 'Track_17',
 'Track_18',
 'Track_19',
 'TrackPosition'

In [34]:
X = df.drop(columns=['Acceleration', 'Braking','Clutch', 'Gear', 'Steering'])
y = df[['Acceleration', 'Braking','Clutch', 'Steering']]


In [35]:
# Normalize features using StandardScaler and save scaling parameters for later use
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled  

os.makedirs('models', exist_ok=True)

np.save(os.path.join('models', 'stds.npy'), scaler.scale_)
np.save(os.path.join('models', 'means.npy'), scaler.mean_)

In [36]:
# Split the scaled data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((141149, 73), (35288, 73), (141149, 4), (35288, 4))

In [37]:
# Define and compile a deep neural network model for multi-output regression using ReLU activations and dropout

model = Sequential()
model.add(Dense(512, input_dim=73, activation='relu',kernel_initializer='he_normal'))
model.add(Dense(128, activation='relu',kernel_initializer='he_normal'))
model.add(Dense(64, activation='relu',kernel_initializer='he_normal'))
model.add(Dropout(0.2))  
# Dropout layer to prevent overfitting
model.add(Dense(32, activation='relu',kernel_initializer='he_normal'))
#Output layer with 3 neurons for 3 outputs
model.add(Dense(4, activation='linear',kernel_initializer='he_normal'))

model.compile(optimizer=Adam(learning_rate=0.007), loss='mean_squared_error', metrics=['mae'])

c:\Users\hamma\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [38]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 512)            │        37,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 114,020 (445.39 KB)

 Trainable params: 114,020 (445.39 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
# Train the model on the training data and validate on the test set over 100 epochs
model.fit(X_train, y_train,validation_data=(X_test,y_test), epochs=200, batch_size=256, verbose=1)

Epoch 1/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0168 - mae: 0.0677 - val_loss: 0.0150 - val_mae: 0.0661
Epoch 2/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0143 - mae: 0.0584 - val_loss: 0.0148 - val_mae: 0.0654
Epoch 3/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0130 - mae: 0.0541 - val_loss: 0.0139 - val_mae: 0.0618
Epoch 4/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0127 - mae: 0.0530 - val_loss: 0.0130 - val_mae: 0.0614
Epoch 5/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0120 - mae: 0.0512 - val_loss: 0.0120 - val_mae: 0.0554
Epoch 6/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0117 - mae: 0.0507 - val_loss: 0.0124 - val_mae: 0.0566
Epoch 7/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0115 - mae: 0.0498 - val_loss: 0.0108 - val_mae: 0.0512
Epoch 8/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0111 - mae: 0.0486 - val_loss: 0.0106 - val_mae: 0.0465
Epoch 9/200
552/552 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms

In [ ]:
# Save the trained model in both Keras and HDF5 formats
import os
import joblib
from tensorflow.keras.models import load_model

# Create a models directory if it doesn't exist
model_dir = os.path.join(os.path.dirname(os.getcwd()), "models")
os.makedirs(model_dir, exist_ok=True)

# Save the model using the newer Keras format
model_path = os.path.join(model_dir, "torcs_driver_model.keras")
model.save(model_path)
print(f"Model saved to {model_path}")

# Save the model using the older HDF5 format
hdf5_model_path = os.path.join(model_dir, "torcs_driver_model.h5")
model.save(hdf5_model_path)
print(f"Model saved to {hdf5_model_path}")

# Save the scaler using joblib
scaler_path = os.path.join(model_dir, "torcs_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

Model saved to c:\Users\hamma\Desktop\TESTING\TORCS-Using-NeuralNetworks-main (2)\TORCS-Using-NeuralNetworks-main\models\torcs_driver_model.keras
Model saved to c:\Users\hamma\Desktop\TESTING\TORCS-Using-NeuralNetworks-main (2)\TORCS-Using-NeuralNetworks-main\models\torcs_driver_model.h5
Scaler saved to c:\Users\hamma\Desktop\TESTING\TORCS-Using-NeuralNetworks-main (2)\TORCS-Using-NeuralNetworks-main\models\torcs_scaler.joblib


In [ ]:
# Compare predictions from the original and reloaded model to verify consistency after saving/loading
prediction = model.predict(X_test[10100].reshape(1, -1))

loaded_model = load_model(model_path)
loaded_scaler = joblib.load(scaler_path)



prediction_loaded = loaded_model.predict(X_test[10100].reshape(1,-1), verbose=0)

print("Prediction with loaded model:")
print("Acceleration:", prediction_loaded[0][0])
print("Braking:", prediction_loaded[0][1])
print("Steering:", prediction_loaded[0][2])


print("\nOriginal prediction:")
print("Acceleration:", prediction[0][0])
print("Braking:", prediction[0][1])
print("Steering:", prediction[0][2])


is_close = np.allclose(prediction_loaded, prediction, rtol=1e-5)
print(f"\nPredictions match: {is_close}")
if not is_close:
    print("\nDifference:")
    print(np.abs(prediction_loaded - prediction))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


c:\Users\hamma\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Prediction with loaded model:
Acceleration: 0.7268384
Braking: 4.2186885
Steering: -0.90245855

Original prediction:
Acceleration: 0.7268384
Braking: 4.2186885
Steering: -0.90245855

Predictions match: True
